In [1]:
# =========================================================
# MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================================================
# INSTALL LIBRARIES
# =========================================================

!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.0 MB/s eta 0:00:00


In [3]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported")

Libraries Imported


In [4]:
# =========================================================
# LOAD FEATURE ENGINEERED DATASETS
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'

test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

Train Shape : (77299, 37)
Test Shape  : (41778, 36)


In [5]:
# =========================================================
# CHECK GPU
# =========================================================

!nvidia-smi

Fri May 29 00:04:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# =========================================================
# PREPARE FEATURES AND TARGET
# =========================================================

X = train_df.drop(columns=['demand'])

y = train_df['demand']

print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

Feature Shape : (77299, 36)
Target Shape  : (77299,)


In [7]:
# =========================================================
# IDENTIFY CATEGORICAL FEATURES
# =========================================================

categorical_features = [
    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geohash_4',
    'geohash_5',
    'weather_temp_interaction'
]

print(categorical_features)

['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geohash_4', 'geohash_5', 'weather_temp_interaction']


In [8]:
# =========================================================
# TRAIN VALIDATION SPLIT
# =========================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train Shape :", X_train.shape)
print("Validation Shape :", X_valid.shape)

Train Shape : (61839, 36)
Validation Shape : (15460, 36)


In [9]:
# =========================================================
# INITIALIZE CATBOOST
# =========================================================

model = CatBoostRegressor(

    iterations=3000,

    learning_rate=0.03,

    depth=8,

    loss_function='RMSE',

    eval_metric='R2',

    random_seed=42,

    task_type='GPU',

    devices='0',

    verbose=200
)

In [10]:
# =========================================================
# TRAIN MODEL
# =========================================================

model.fit(

    X_train,
    y_train,

    cat_features=categorical_features,

    eval_set=(X_valid, y_valid),

    use_best_model=True
)

Default metric period is 5 because R2 is/are not implemented for GPU
Metric R2 is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.0515557	test: 0.0516731	best: 0.0516731 (0)	total: 199ms	remaining: 9m 57s
200:	learn: 0.9380306	test: 0.9342771	best: 0.9342771 (200)	total: 9.18s	remaining: 2m 7s
400:	learn: 0.9475022	test: 0.9417753	best: 0.9417753 (400)	total: 13.6s	remaining: 1m 27s
600:	learn: 0.9532821	test: 0.9457448	best: 0.9457448 (600)	total: 20.7s	remaining: 1m 22s
800:	learn: 0.9570750	test: 0.9481059	best: 0.9481059 (800)	total: 25.2s	remaining: 1m 9s
1000:	learn: 0.9600479	test: 0.9498624	best: 0.9498624 (1000)	total: 29.9s	remaining: 59.6s
1200:	learn: 0.9622610	test: 0.9510494	best: 0.9510494 (1200)	total: 37.1s	remaining: 55.5s
1400:	learn: 0.9640596	test: 0.9519257	best: 0.9519257 (1400)	total: 41.6s	remaining: 47.5s
1600:	learn: 0.9657220	test: 0.9527031	best: 0.9527031 (1600)	total: 49s	remaining: 42.8s
1800:	learn: 0.9671275	test: 0.9532579	best: 0.9532579 (1800)	total: 53.6s	remaining: 35.7s
2000:	learn: 0.9683848	test: 0.9538265	best: 0.9538265 (2000)	total: 58.2s	remaining: 29.1s
2

CatBoostRegressor(depth=8, devices='0', eval_metric='R2', iterations=3000, learning_rate=0.03, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=200)

In [11]:
# =========================================================
# VALIDATION PREDICTIONS
# =========================================================

valid_preds = model.predict(X_valid)

In [12]:
# =========================================================
# COMPUTE R² SCORE
# =========================================================

r2 = r2_score(y_valid, valid_preds)

print("Validation R² Score :", r2)

print("Competition Score :", max(0, 100 * r2))

Validation R² Score : 0.955412268699826
Competition Score : 95.5412268699826


In [13]:
# =========================================================
# FEATURE IMPORTANCE
# =========================================================

feature_importance = pd.DataFrame({

    'Feature': X.columns,

    'Importance': model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
30,geohash_demand_mean,36.160344
3,RoadType,14.548200
33,roadtype_demand_mean,10.417826
31,hour_demand_mean,5.625939
21,geohash_5,5.237933
0,Index,4.648818
14,hour_cos,3.884237
1,geohash,3.867460
34,geo_hour_density,3.436004
20,geohash_4,1.828294


In [14]:
# =========================================================
# TEST PREDICTIONS
# =========================================================

test_preds = model.predict(test_df)

In [15]:
# =========================================================
# CREATE SUBMISSION
# =========================================================

submission = pd.DataFrame({

    'Index': test_df['Index'],

    'demand': test_preds
})

submission.head()

,Index,demand
0,0,0.044697
1,1,0.023995
2,2,0.017696
3,3,0.023325
4,4,0.042568


In [16]:
# =========================================================
# SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission_catboost.csv'

submission.to_csv(submission_path, index=False)

print("Submission Saved Successfully")

Submission Saved Successfully


In [17]:
# =========================================================
# TRAFFIC DEMAND PREDICTION - FINAL CATBOOST TRAINING
# =========================================================

# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# 2. INSTALL LIBRARIES
# =========================================================

!pip install catboost -q

# =========================================================
# 3. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

print("Libraries Imported Successfully")

# =========================================================
# 4. LOAD FEATURE ENGINEERED DATASETS
# =========================================================

train_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/train_feature_engineered.csv'

test_path = '/content/drive/MyDrive/Traffic_Prediction/dataset/test_feature_engineered.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train Shape :", train_df.shape)
print("Test Shape  :", test_df.shape)

# =========================================================
# 5. CHECK GPU
# =========================================================

!nvidia-smi

# =========================================================
# 6. REMOVE UNWANTED COLUMNS
# =========================================================

# Remove Index because it may cause leakage
# Remove demand from test if present

if 'Index' in train_df.columns:
    train_df.drop(columns=['Index'], inplace=True)

if 'Index' in test_df.columns:
    test_df.drop(columns=['Index'], inplace=True)

if 'demand' in test_df.columns:
    test_df.drop(columns=['demand'], inplace=True)

print("Unwanted Columns Removed")

# =========================================================
# 7. PREPARE FEATURES AND TARGET
# =========================================================

X = train_df.drop(columns=['demand'])

y = train_df['demand']

print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

# =========================================================
# 8. IDENTIFY CATEGORICAL FEATURES
# =========================================================

categorical_features = [

    'geohash',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'geohash_4',
    'geohash_5',
    'weather_temp_interaction'
]

print("Categorical Features:")
print(categorical_features)

# =========================================================
# 9. TRAIN VALIDATION SPLIT
# =========================================================

X_train, X_valid, y_train, y_valid = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42
)

print("Train Shape :", X_train.shape)
print("Validation Shape :", X_valid.shape)

# =========================================================
# 10. CONVERT CATEGORICAL FEATURES TO STRING
# =========================================================

for col in categorical_features:

    X_train[col] = X_train[col].astype(str)

    X_valid[col] = X_valid[col].astype(str)

    test_df[col] = test_df[col].astype(str)

print("Categorical Conversion Completed")

# =========================================================
# 11. INITIALIZE CATBOOST MODEL
# =========================================================

model = CatBoostRegressor(

    iterations=5000,

    learning_rate=0.03,

    depth=8,

    loss_function='RMSE',

    eval_metric='RMSE',

    random_seed=42,

    task_type='GPU',

    devices='0',

    verbose=200
)

print("CatBoost Initialized")

# =========================================================
# 12. TRAIN MODEL
# =========================================================

model.fit(

    X_train,
    y_train,

    cat_features=categorical_features,

    eval_set=(X_valid, y_valid),

    use_best_model=True,

    early_stopping_rounds=300
)

# =========================================================
# 13. VALIDATION PREDICTIONS
# =========================================================

valid_preds = model.predict(X_valid)

# =========================================================
# 14. COMPUTE R² SCORE
# =========================================================

r2 = r2_score(y_valid, valid_preds)

competition_score = max(0, 100 * r2)

print("\n")
print("="*50)
print("VALIDATION RESULTS")
print("="*50)

print(f"Validation R² Score : {r2}")

print(f"Competition Score   : {competition_score}")

# =========================================================
# 15. FEATURE IMPORTANCE
# =========================================================

feature_importance = pd.DataFrame({

    'Feature': X.columns,

    'Importance': model.feature_importances_
})

feature_importance = feature_importance.sort_values(

    by='Importance',

    ascending=False
)

print("\n")
print("="*50)
print("TOP 20 FEATURE IMPORTANCE")
print("="*50)

print(feature_importance.head(20))

# =========================================================
# 16. TEST PREDICTIONS
# =========================================================

test_preds = model.predict(test_df)

# =========================================================
# 17. CREATE SUBMISSION FILE
# =========================================================

# Reload original test to get Index

original_test = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Prediction/dataset/test.csv'
)

submission = pd.DataFrame({

    'Index': original_test['Index'],

    'demand': test_preds
})

print("\n")
print("="*50)
print("SUBMISSION SAMPLE")
print("="*50)

print(submission.head())

# =========================================================
# 18. SAVE SUBMISSION
# =========================================================

submission_path = '/content/drive/MyDrive/Traffic_Prediction/submission1_catboost_final.csv'

submission.to_csv(

    submission_path,

    index=False
)

print("\n")
print("="*50)
print("SUBMISSION SAVED SUCCESSFULLY")
print("="*50)

print(f"Saved At : {submission_path}")

# =========================================================
# 19. OPTIONAL - SAVE FEATURE IMPORTANCE
# =========================================================

feature_importance_path = '/content/drive/MyDrive/Traffic_Prediction/feature_importance.csv'

feature_importance.to_csv(

    feature_importance_path,

    index=False
)

print("\nFeature Importance Saved Successfully")

# =========================================================
# 20. FINAL SUMMARY
# =========================================================

print("\n")
print("="*60)
print("MODEL TRAINING COMPLETED SUCCESSFULLY")
print("="*60)

print(f"Final Validation R² : {r2:.6f}")

print(f"Competition Score   : {competition_score:.6f}")

print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries Imported Successfully
Train Shape : (77299, 37)
Test Shape  : (41778, 36)
Fri May 29 00:16:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0  